<a href="https://colab.research.google.com/github/ahmadtariq2004/Evolvix-Ai-ML-Internship-ahmadtariq2004/blob/main/week-4/task-1/week4_task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 Task 1: Sentiment Analysis REST API

The Week 3 notebook uses the Hugging Face checkpoint `ahmed792002/Finetuning_Longformer_IMDb_movie_reviews_Classification` but does not save it locally. This notebook downloads that same checkpoint once if needed, saves the tokenizer and model into the `sentiment_model/` folder beside this notebook, and then loads the API model from that local folder. Run the cells from top to bottom.

The first run requires an internet connection to download the checkpoint; later runs use the local copy.

In [ ]:
%pip install -q fastapi uvicorn transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

MODEL_ID = "ahmed792002/Finetuning_Longformer_IMDb_movie_reviews_Classification"
MODEL_DIR = Path.cwd() / "sentiment_model"

if not (MODEL_DIR / "config.json").is_file():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {MODEL_ID} and saving it to {MODEL_DIR}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
    tokenizer.save_pretrained(MODEL_DIR)
    model.save_pretrained(MODEL_DIR)

sentiment_pipeline = pipeline(
    task="text-classification",
    model=str(MODEL_DIR),
    tokenizer=str(MODEL_DIR),
    local_files_only=True,
)
print(f"Sentiment model loaded locally from {MODEL_DIR}")

c:\Users\ASJAL\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c:\Users\ASJAL\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASJAL\.cache\huggingface\hub\models--ahmed792002--Finetuning_Longformer_IMDb_movie_reviews_Classification. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 273/273 [00:

Sentiment model loaded locally from c:\Users\ASJAL\Downloads\WEEK 4 TASK 1\sentiment_model


## Test with Postman

Run the API cell below, then send this request while the server is running:

- Method: `POST`
- URL: `http://127.0.0.1:8000/predict`
- Header: `Content-Type: application/json`
- Body: **raw** → **JSON**

```json
{
  "text": "This movie was absolutely fantastic and I loved the acting!"
}
```

A successful response contains the model's label and confidence score, for example `{
:
,
:0.99}`. The label names are supplied by the loaded checkpoint. Stop the server from another cell with `api_server.should_exit = True`.

In [ ]:
import threading

import uvicorn
from fastapi import FastAPI, HTTPException, Request

app = FastAPI(title="IMDb Sentiment Analysis API")


@app.get("/")
def health_check():
    return {"status": "ready", "model": MODEL_ID}


@app.post("/predict")
async def predict(request: Request):
    try:
        payload = await request.json()
    except Exception as error:
        raise HTTPException(
            status_code=400,
            detail="Request body must be valid JSON.",
        ) from error

    if not isinstance(payload, dict):
        raise HTTPException(status_code=400, detail="JSON body must be an object.")

    text = payload.get("text")
    if not isinstance(text, str) or not text.strip():
        raise HTTPException(
            status_code=400,
            detail="Provide a non-empty string in the 'text' field.",
        )

    try:
        result = sentiment_pipeline(text.strip())[0]
        return {"sentiment": result["label"], "score": float(result["score"])}
    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail="Sentiment prediction failed. Check the model and input.",
        ) from error


if "api_server_thread" in globals() and api_server_thread.is_alive():
    print("API server is already running at http://127.0.0.1:8000")
else:
    api_config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
    api_server = uvicorn.Server(api_config)
    api_server_thread = threading.Thread(target=api_server.run, daemon=True)
    api_server_thread.start()
    print("API server starting at http://127.0.0.1:8000")

API server is already running at http://127.0.0.1:8000
